# Obnoxious _p_-Median Problem

## Problem definition

In the broader topic of facility location disparsing, obnoxious facilities were not taken into account when modelling such problems. Obnoxious facilities includes nuclear power plants, waste dumpsites and other structures that negatively impact clients, the environment and other facilities. The problem discussed in this notebook handles such cases, trying to maximize the distance between obnoxious plants and other facilities.

## The unified model
Based on the method of Ogryczak and Tamir for minimizing the sum of the K largest 
functions, Lei and Church proposed a unified dispersion model where distances are 
measured between facilities. The adaptation to the OpMP setting requires a key 
conceptual shift: distances are now measured from **clients to facilities** rather 
than between facilities. This change is reflected both in the index set (I instead 
of J for the outer variables) and in the coefficient of $s_i$, which becomes L instead 
of L+1, since client i is not itself a facility and contributes no zero self-distance.

### Parameters and Variables

**Sets:**
- $I = \{1, \dots, n\}$: set of clients
- $J = \{1, \dots, m\}$: set of potential facility sites

**Parameters:**
- $d(i, j)$: distance between client $i \in I$ and facility $j \in J$
- $p$: number of facilities to open
- $K$: number of clients considered in the objective (set to $n$ for the OpMP)
- $L$: number of closest facilities considered per client (set to $1$ for the OpMP)
- $M$: a sufficiently large constant

**Decision variables:**
- $y_j \in \{0,1\}$: equals $1$ if facility $j$ is opened, $0$ otherwise

**Auxiliary variables:**
- $t \in \mathbb{R}$: threshold value used to identify the K smallest partial sums
- $u_i \geq 0$: excess of client $i$'s contribution over the threshold $t$
- $q_i \geq 0$: intermediate variable representing the weighted sum of distances 
  for client $i$, computed from $s_i$ and $z(i,j)$
- $s_i \geq 0$: plays a role analogous to $t$, but at the client level
- $z(i,j) \geq 0$: non-zero when facility $j$ is among the $L$ closest to client $i$

### Model Formulation

$$\max \; Kt - \sum_{i \in I} u_i \tag{12}$$

**Subject to:**

$$t - u_i \leq q_i \qquad \forall i \in I \tag{13}$$

$$q_i = Ls_i - \sum_{j \in J} z(i,j) \qquad \forall i \in I \tag{14}$$

$$s_i - z(i,j) \leq d(i,j) + M(1 - y_j) \qquad \forall i \in I, \forall j \in J \tag{15}$$

$$\sum_{j \in J} y_j = p \tag{3}$$

$$u_i, z(i,j) \geq 0, \quad y_j \in \{0,1\} \qquad \forall i \in I, \forall j \in J$$

**Intuition behind each constraint:**

- **(12)** — The objective maximizes the sum of the $K$ smallest client-to-facility 
  distances. With $K=n$ this reduces to maximizing the sum over all clients, 
  which is exactly the OpMP objective.

- **(13)** — Links $u_i$ to the threshold $t$: if client $i$ is among the $K$ 
  worst-served, $u_i = 0$ and the full contribution $q_i$ is counted; 
  otherwise $u_i$ absorbs the excess.

- **(14)** — Defines $q_i$ as the weighted minimum distance from client $i$ 
  to the $L$ nearest open facilities. With $L=1$ this simplifies to the 
  distance from client $i$ to its single nearest open facility.

- **(15)** — The big-M constraint: if facility $j$ is open ($y_j = 1$), 
  forces $s_i - z(i,j) \leq d(i,j)$, effectively capturing the distance 
  from client $i$ to facility $j$ in the auxiliary variables.

- **(3)** — Exactly $p$ facilities must be opened.

### Implementation

In [18]:
import gurobipy as gp
from gurobipy import GRB

#### Mapping to the Gurobi's implementation
To implement the above model, I define the following function:

```python
solve_unified_opmp(n, m, p, d, K=None, L=1, M=None)
```

with **Parameters**:
```python
n: int   # number of clients (|I|)
m: int   # number of potential facility sites (|J|)
p: int   # number of facilities to open
d: list[list[float]] # distance matrix, d[i][j] = distance from client i to facility j
K: int   # clients counted in the objective; defaults to n (full OpMP)
L: int   # closest facilities considered per client; defaults to 1 (full OpMP)
M: float # big-M constant; defaults to max distance in d
```
and **Returns a dictionary** with keys:
```python
"obj": model.ObjVal, # optimal objective value
"open": open_facilities, # list of opened facility indices
"assignment": assignment, # dict mapping each client i to its nearest open facility
"model": model, # the solved Gurobi Model object
```

In [ ]:
def solve_unified_opmp(n, m, p, d, K=None, L=1, M=None):

    """
    I decided to preserve the original modular structure of the problem, in which one can redefine K and L.

    Here I want to stick with K = n and L = 1 to implement the intended OpMP model:
    - K = n to maximize the smallest distances from the facilities of all the clients
    - L = 1 to consider the distances of the other facilities from a client, only in relation to the nearest one
    """

    if K is None:
        K = n  # with K=n the model is equivalent to the compact OpMP
        
    if M is None:
        M = max(d[i][j] for i in range(n) for j in range(m))


    I = range(n)  # client index set
    J = range(m)  # facility index set


    model = gp.Model("unified_opmp")


    # Decisional variables

    # y[j] = 1 if facility j is opened
    y = model.addVars(J, vtype=GRB.BINARY, name="y")


    # Auxiliary variables

    # t: global threshold identifying the K smallest partial sums q_i
    t = model.addVar(lb=-GRB.INFINITY, name="t")

    # u[i]: non-negative slack absorbing the excess of q_i over t;
    #       zero when client i is among the K worst-served
    u = model.addVars(I, lb=0.0, name="u")

    # s[i]: client-level threshold, analogous to t but local to client i;
    #       drives the big-M constraint (15) to capture distances
    s = model.addVars(I, lb=0.0, name="s")

    # z[i,j]: non-zero when facility j is among the L nearest open facilities
    #         to client i; together with s[i] it encodes the L-th nearest distance
    z = model.addVars(I, J, lb=0.0, name="z")

    # q[i]: intermediate variable; equals L*s[i] - sum_j z[i,j],
    #       which resolves to the distance from client i to its nearest open
    #       facility when L=1
    q = model.addVars(I, lb=0.0, name="q")


    # Objective function

    # Maximise K*t - sum_i u[i], which equals the sum of the K smallest q_i
    model.setObjective(
        K * t - gp.quicksum(u[i] for i in I),
        GRB.MAXIMIZE
    )


    # Constraint: t - u[i] <= q[i]  for all i 
    
    # Ensures u[i] captures the excess of q[i] over t;
    # clients with q[i] >= t contribute t to the sum (u[i]=0),
    # while those below contribute their actual q[i] (u[i] = t - q[i])
    model.addConstrs(
        (t - u[i] <= q[i] for i in I),
        name="link_u_q"
    )


    # Constraint: q[i] = L*s[i] - sum_j z[i,j]  for all i 

    # Defines q[i] as the telescopic difference between the client threshold s[i]
    # and the total slack allocated to the L nearest facilities;
    # with L=1 this collapses to q[i] = s[i] - z[i, nearest_j]
    model.addConstrs(
        (q[i] == L * s[i] - gp.quicksum(z[i, j] for j in J) for i in I),
        name="def_q"
    )


    # Constraint: s[i] - z[i,j] <= d[i][j] + M*(1-y[j])  

    # Big-M constraint: active only when facility j is open (y[j]=1);
    # forces s[i] - z[i,j] <= d[i][j], linking the auxiliary variables
    # to the actual client-to-facility distance
    model.addConstrs(
        (s[i] - z[i, j] <= d[i][j] + M * (1 - y[j]) for i in I for j in J),
        name="bigM"
    )


    # Constraint: sum_j y[j] = p  

    # Exactly p facilities must be opened
    model.addConstr(
        gp.quicksum(y[j] for j in J) == p,
        name="cardinality"
    )


    model.optimize()


    # Solution extraction
    open_facilities = [j for j in J if y[j].X > 0.5]


    # Recover the nearest open facility for each client from the distance matrix
    assignment = {
        i: min(open_facilities, key=lambda j: d[i][j])
        for i in I
    }
    

    return {
        "obj": model.ObjVal,
        "open": open_facilities,
        "assignment": assignment,
        "model": model,
    }

### Loading benchmark instances

The Belotti repository provides instances in two formats:

- **`euclidean`** — 2D Cartesian coordinates for clients and facilities; distances are computed as Euclidean.
- **`table`** — a pre-computed $n \times m$ distance matrix.

Files are either plain text (`.euc`, `.table`) or gzip-compressed (`.gz`). The parser below handles both transparently.

In [20]:
import gzip
import math
import re


def _open_instance(path: str):
    """Return the full text of an instance file, decompressing .gz if needed."""
    if path.endswith(".gz"):
        with gzip.open(path, "rt") as f:
            return f.read()
    with open(path) as f:
        return f.read()


def _parse_coords(block: str) -> list:
    """
    Extract (x, y) pairs from the coordinate block used in euclidean files.
    Both formats are supported:
      name {x, y}          (old .euc style)
      name\t{x, y},        (new .gz euclidean style)
    """
    return [
        (float(x), float(y))
        for x, y in re.findall(r"\{\s*([\d.]+)\s*,\s*([\d.]+)\s*\}", block)
    ]


def _parse_table(text: str, n: int, m: int) -> list:
    """
    Extract the distance matrix from the 'table = ...' block and return it
    as a (n x m) list-of-lists (rows = clients, columns = facilities).

    Two layout variants exist in the dataset:
      - Normal  (n rows, m values each) — old plain-text files and orlib .gz.
      - Transposed (m rows, n values each) — most .gz files (Hoefer, BildeKrarup ...).
    The function auto-detects the layout and transposes when necessary.
    """
    # Capture everything between the first '{' after 'table =' and the last '}'
    match = re.search(r"(?i)table\s*=\s*\{(.*)\}\s*$", text, re.DOTALL)
    if match is None:
        raise ValueError("Could not find 'table' block")

    inner = match.group(1).strip()

    # Strip one leading '{' and one trailing '}' produced by the double-brace
    # variant used in old plain-text files (table = {{ ... }})
    if inner.startswith("{"):
        inner = inner[1:]
    if inner.endswith("}"):
        inner = inner[:-1]

    # Split into rows on the },{ boundary (tolerates whitespace/newlines)
    rows = re.split(r"\}\s*,\s*\{", inner)
    d = [[float(v) for v in re.findall(r"[\d.]+", row)] for row in rows]

    # Auto-transpose: some files store the matrix as (m x n) instead of (n x m)
    if len(d) == m and len(d[0]) == n:
        d = [[d[j][i] for j in range(m)] for i in range(n)]

    return d


def load_instance(path: str) -> dict:
    """
    Parse a Belotti-format OpMP instance (plain text or .gz).

    Returns a dict with:
        'n'  : number of clients
        'm'  : number of potential facility sites
        'p'  : number of facilities to open
        'd'  : distance matrix as list[list[float]], shape (n, m)
    """
    text = _open_instance(path)

    # --- read scalar parameters (case-insensitive) ---------------------------
    def _int(key):
        match = re.search(rf"(?i)(?<![a-zA-Z]){key}\s*=\s*(\d+)", text)
        if match is None:
            raise ValueError(f"Parameter '{key}' not found in {path}")
        return int(match.group(1))

    n = _int("n")
    m = _int("m")
    p = _int("p")
    fmt = re.search(r"(?i)type\s*=\s*(\w+)", text).group(1).lower()

    if fmt == "euclidean":
        # --- euclidean: parse coordinate blocks, then compute distances ------
        # Split on the 'facilities' keyword to separate the two blocks
        split = re.split(r"(?i)facilities\s*=", text, maxsplit=1)
        client_block   = split[0]
        facility_block = split[1]

        clients    = _parse_coords(client_block)
        facilities = _parse_coords(facility_block)

        d = [
            [math.hypot(cx - fx, cy - fy) for (fx, fy) in facilities]
            for (cx, cy) in clients
        ]

    else:
        d = _parse_table(text, n, m)

    return {"n": n, "m": m, "p": p, "d": d}


### Testing on benchmark instances

We run the unified model on a selection of instances from the Belotti dataset, chosen to cover different sizes and formats. Results can be compared against Table 1 of the paper.

In [21]:
import os

# Base path to the extracted repository

BASE = "instances"

# Instances to test: (label, path)
# I pick small-to-medium ones where the paper reports finite solve times

TINY_TEST_INSTANCES = [
    # orlib: tiny, good for smoke-testing the parser and solver

    ("orlib-cap41",  os.path.join(BASE, "data", "orlib-cap41.gz")),
    ("orlib-cap81",  os.path.join(BASE, "data", "orlib-cap81.gz")),
    ("orlib-cap111", os.path.join(BASE, "data", "orlib-cap111.gz"))
]

MEDIUM_TEST_INSTANCES = [
    # BildeKrarup-D: 80 clients, 30 facilities — moderate size

    ("BK-D1.1", os.path.join(BASE, "data-tr", "BildeKrarup-Dq-1-D1.1.gz")),
    ("BK-D2.1", os.path.join(BASE, "data-tr", "BildeKrarup-Dq-2-D2.1.gz"))
]

# Hoefer-O: 100 clients, 100 facilities — larger

LARGE_TEST_INSTANCE = ("Hoefer-O1", os.path.join(BASE, "data", "Hoefer-O-MO1.gz"))

In [22]:
# Tiny test instances

for label, path in TINY_TEST_INSTANCES:
    inst = load_instance(path)
    n, m, p, d = inst["n"], inst["m"], inst["p"], inst["d"]
    print(f"\n{'='*60}")
    print(f"Instance : {label}  (n={n}, m={m}, p={p})")
    print(f"{'='*60}")

    sol = solve_unified_opmp(n, m, p, d)


Instance : orlib-cap41  (n=50, m=16, p=4)
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (linux64 - "CachyOS")

CPU model: AMD Ryzen 7 9700X 8-Core Processor, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 901 rows, 967 columns and 3466 nonzeros (Max)
Model fingerprint: 0xa209407b
Model has 51 linear objective coefficients
Variable types: 951 continuous, 16 integer (16 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+06]
  Objective range  [1e+00, 5e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [4e+00, 3e+06]

Found heuristic solution: objective -0.0000000
Presolve removed 50 rows and 850 columns
Presolve time: 0.00s
Presolved: 851 rows, 117 columns, 2516 nonzeros
Variable types: 100 continuous, 17 integer (16 binary)

Root relaxation: objective 5.189686e+07, 179 iterations, 0.00 seconds (0.01 work units)

    Nodes    |    Current Node    |     Objective Bounds     

In [23]:
# Medium test instances

for label, path in MEDIUM_TEST_INSTANCES:
    inst = load_instance(path)
    n, m, p, d = inst["n"], inst["m"], inst["p"], inst["d"]
    print(f"\n{'='*60}")
    print(f"Instance : {label}  (n={n}, m={m}, p={p})")
    print(f"{'='*60}")

    sol = solve_unified_opmp(n, m, p, d)


Instance : BK-D1.1  (n=80, m=30, p=7)
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (linux64 - "CachyOS")

CPU model: AMD Ryzen 7 9700X 8-Core Processor, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 2561 rows, 2671 columns and 10030 nonzeros (Max)
Model fingerprint: 0x35b17912
Model has 81 linear objective coefficients
Variable types: 2641 continuous, 30 integer (30 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+03]
  Objective range  [1e+00, 8e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [7e+00, 2e+03]

Found heuristic solution: objective -0.0000000
Presolve removed 80 rows and 2480 columns
Presolve time: 0.01s
Presolved: 2481 rows, 191 columns, 7387 nonzeros
Variable types: 81 continuous, 110 integer (30 binary)

Root relaxation: objective 6.373900e+04, 271 iterations, 0.01 seconds (0.02 work units)

    Nodes    |    Current Node    |     Objective Bounds   

In [24]:
# Large test instance

label = LARGE_TEST_INSTANCE[0]
path = LARGE_TEST_INSTANCE[1]

inst = load_instance(path)
n, m, p, d = inst["n"], inst["m"], inst["p"], inst["d"]
print(f"\n{'='*60}")
print(f"Instance : {label}  (n={n}, m={m}, p={p})")
print(f"{'='*60}")

sol = solve_unified_opmp(n, m, p, d)


Instance : Hoefer-O1  (n=100, m=100, p=25)
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (linux64 - "CachyOS")

CPU model: AMD Ryzen 7 9700X 8-Core Processor, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 10201 rows, 10401 columns and 40600 nonzeros (Max)
Model fingerprint: 0x2c2a0989
Model has 101 linear objective coefficients
Variable types: 10301 continuous, 100 integer (100 binary)
Coefficient statistics:
  Matrix range     [1e+00, 5e+01]
  Objective range  [1e+00, 1e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [2e+01, 1e+02]

Found heuristic solution: objective -0.0000000
Presolve removed 100 rows and 10100 columns
Presolve time: 0.04s
Presolved: 10101 rows, 301 columns, 30300 nonzeros
Variable types: 201 continuous, 100 integer (100 binary)
Root relaxation presolved: 10101 rows, 301 columns, 30300 nonzeros


Root relaxation: objective 3.990153e+03, 641 iterations, 0.05 s